# Agent 4 — Gather Everything, Keep the Receipts

The angles fan out into searches, the searches into fetches, the fetched
pages into **notes** — and every note carries its source URL and date from
the moment it's written. This whole notebook runs offline.

In [ ]:
# The mini-web: nine pages about the (fictional) Riverside Community Garden.
# Small enough to read whole, real enough to research. One page is wrong on purpose.
MINIWEB = {
 "riverside-garden.org/about": {"date": "2026-05-10", "title": "Our garden today",
  "text": "The Riverside Community Garden has 60 plots and 48 member families. "
          "We grow vegetables for members and donate surplus to the food pantry."},
 "riverside-garden.org/history": {"date": "2023-05-02", "title": "Our history",
  "text": "Founded in 2019 with a dozen beds. The sign by the gate lists 48 plots, "
          "painted when we finished the 2023 season."},
 "riverside-garden.org/join": {"date": "2026-06-01", "title": "Join us",
  "text": "Want a plot? The waitlist currently holds 22 families. Members pay a "
          "small annual fee and share watering duties."},
 "lakeview-news.com/garden-expands": {"date": "2026-04-20", "title": "Garden adds 12 plots",
  "text": "The Riverside Community Garden completed its expansion this spring, "
          "taking the garden from 48 plots to 60. Organizers credit a city grant."},
 "lakeview-news.com/roundup-2023": {"date": "2023-09-15", "title": "Community roundup",
  "text": "At the Riverside garden, 31 member families closed out the 2023 season "
          "with a harvest festival."},
 "cityparks.gov/report-2026": {"date": "2026-03-14", "title": "Community garden census",
  "text": "Riverside Community Garden: 60 plots, 48 member families, established "
          "2019. Census conducted March 2026."},
 "cityparks.gov/grants-2025": {"date": "2025-11-08", "title": "2025 grant awards",
  "text": "Riverside Community Garden: $15,000 for expansion. The site's land "
          "lease with the parks department runs through 2028."},
 "gardenblog.example.com/visit": {"date": "2026-02-02", "title": "A visit to Riverside",
  "text": "Lovely afternoon at Riverside! I heard they have 600 plots now, which "
          "explains the crowds. The tomatoes were spectacular."},
 "gardenblog.example.com/opinion": {"date": "2026-01-05", "title": "Why gardens matter",
  "text": "Community gardens are the beating heart of a neighborhood. Riverside "
          "is a treasure and everyone loves it."},
}

import re as _re, collections as _c
def _words(text):
    return set(w for w in _re.findall(r"[a-z0-9]+", text.lower()) if len(w) > 2)
_DF = _c.Counter()                       # in how many pages does each word appear?
for _p in MINIWEB.values():
    for _w in _words(_p["title"] + " " + _p["text"]):
        _DF[_w] += 1

def search(query):
    """Score pages by shared words, each weighted by rarity (1/pages-containing-it).
    'riverside' is on every page and says nothing; 'waitlist' is on one and says a lot."""
    qwords = _words(query)
    scored = []
    for url, page in MINIWEB.items():
        shared = qwords & _words(page["title"] + " " + page["text"])
        scored.append((sum(1.0 / _DF[w] for w in shared), url, page["title"]))
    scored.sort(reverse=True)
    return [(url, title) for score, url, title in scored[:3] if score > 0.3]

def fetch(url):
    """Return a page's text with its receipt (url and date) attached."""
    page = MINIWEB[url]
    return {"url": url, "date": page["date"], "text": page["text"]}

print(f"{len(MINIWEB)} pages online.")
print("search('riverside garden plots') ->")
for url, title in search("riverside garden plots"):
    print("  ", url, "-", title)

## Dedupe, fetch once, take notes

Different angles find the same pages. The `seen` set prevents double
fetching — and, worse, one source masquerading as two independent ones
later, when agreement between sources starts to mean something.

In [ ]:
ANGLES = [
    "history of plot numbers at the Riverside garden",
    "Riverside garden member families census",
    "city grant funding for the Riverside garden expansion",
    "waitlist to join the Riverside garden",
]

def gather(angles):
    seen, notes, fetches = set(), [], 0
    for angle in angles:
        for url, title in search(angle):
            if url in seen:
                continue                      # dedupe BEFORE fetching
            seen.add(url)
            page = fetch(url)
            fetches += 1
            # notes = the sentences that might matter (here: any with a digit)
            for sentence in page["text"].split(". "):
                if any(ch.isdigit() for ch in sentence):
                    notes.append({"note": sentence.strip().rstrip("."),
                                  "source": page["url"], "date": page["date"]})
    return notes, fetches, seen

notes, fetches, seen = gather(ANGLES)
print(f"{fetches} pages fetched (each exactly once), {len(notes)} notes taken\n")
for n in notes:
    print(f"  \"{n['note']}\"")
    print(f"      -- {n['source']}  ({n['date']})")

## The stripped copy — feel the law

Same notes, receipts deleted. Now try the downstream task lesson 6 will
need: *the plot count went 48 → 60 — do two independent sources support
that, or one source counted twice?*

In [ ]:
stripped = [{"note": n["note"]} for n in notes]   # receipts gone
print("With receipts - the 48->60 story:")
for n in notes:
    if "48" in n["note"] or "60" in n["note"]:
        print(f"  {n['note']}  [{n['source']} {n['date']}]")
print()
print("Without receipts:")
for n in stripped:
    if "48" in n["note"] or "60" in n["note"]:
        print(f"  {n['note']}  [???]")
print()
print("Same sentences. One version can answer 'independent sources?' - the other never can.")
assert all("source" in n for n in notes), "every note must carry its receipt"

## Try it

1. Comment out the `seen` check and rerun. How many extra fetches? Which
   page now appears as duplicate notes?
2. The note-taker keeps any sentence with a digit. Find a sentence it
   wrongly keeps and one it wrongly drops — the crude filter is lesson 5's
   motivation (the model reads meaning; a rule reads shapes).
3. **Build turn-in:** three notes traced back to their exact source
   sentences — and the one note whose page says something slightly
   different (it's in there; compare dates on the plot counts).